In [ ]:
!pip install langchain

In [ ]:
!pip show langchain

Name: langchain
Version: 1.3.9
Summary: Building applications with LLMs through composability
Home-page: https://docs.langchain.com/
Author: 
Author-email: 
License: MIT
Location: /usr/local/lib/python3.12/dist-packages
Requires: langchain-core, langgraph, pydantic
Required-by: 


create model first

In [ ]:
!pip install -U langchain-google-genai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 2.8 MB/s eta 0:00:00


In [ ]:
from langchain.chat_models import init_chat_model
from google.colab import userdata
from pprint import pprint

google_api_key = userdata.get('GEMINI_API_KEY')
model = init_chat_model(
    "google_genai:gemini-2.5-flash",
    api_key=google_api_key
)

create skilll demand tools

In [ ]:
!pip install langchain-tavily

In [ ]:
from langchain_tavily import TavilySearch
from google.colab import userdata

tavily_api_key = userdata.get('TAVILY_API_KEY')

skill_demand_tool = TavilySearch(
    max_results=5,
    search_depth="advanced",
    tavily_api_key=tavily_api_key
)


create job search tool

In [ ]:
rapidapi_key=userdata.get('RAPIDAPI_KEY')

In [ ]:
import requests
from langchain.tools import tool
from google.colab import userdata

@tool
def search_jobs(skill: str, location: str) -> list:
    """Search for jobs requiring a specific skill and location using JSearch API from RapidAPI."""
    print(f"\nCalling search_jobs tool")
    print(f"Searching jobs for: {skill} jobs in {location}")

    rapidapi_key = userdata.get('RAPIDAPI_KEY')

    url = "https://jsearch.p.rapidapi.com/search-v2"


    querystring = {
        "query": f"{skill} jobs in {location}",
        "page": "1",
        "country": "in",
        "employment_types": "INTERN,FULLTIME",
        "job_requirements": "no_experience,under_3_years_experience"
    }
    headers = {
	  "x-rapidapi-key":rapidapi_key ,
	  "x-rapidapi-host": "jsearch.p.rapidapi.com",
	  "Content-Type": "application/json"
    }

    response = requests.get(url, headers=headers, params=querystring)
    data = response.json()
    # Correctly access the nested list of jobs
    jobs = data.get("data", {}).get("jobs", [])
    print(f"Found {len(jobs)} jobs\n")

    result = []
    for job in jobs:
        result.append({
            "title": job.get("job_title"),
            "company": job.get("employer_name"),
            "location": job.get("job_city"),
            "apply_link": job.get("job_apply_link")
        })
    return result

create an agent

In [ ]:
from langchain.agents import create_agent

In [ ]:
system_prompt = """You are a Skill-to-Career Mapping assistant that helps students understand skill demand and find matching job opportunities.

You have access to these tools:
- skill_demand_tool: Search for industry demand, salary insights, and career trends
- search_jobs: Find actual job listings requiring specific skills

Help the student by researching the skill they ask about and finding relevant opportunities.

Present results in a clean, readable format with clear sections and proper spacing. Include all job details with apply links. Don't use markdown format."""

In [ ]:
from langgraph.checkpoint.memory import InMemorySaver
checkpointer = InMemorySaver()

In [ ]:
agent = create_agent(
    model=model,
    tools=[skill_demand_tool, search_jobs],
    system_prompt=system_prompt,
    checkpointer=checkpointer
)
config={"configurable":{"thread_id":"1"}}

In [ ]:
user_query = "What's the demand for generative ai in the industry and show me related intern job openings in India"

In [ ]:
response = agent.invoke({
    "messages": [{"role": "user", "content": user_query}]
},config=config)


Calling search_jobs tool
Searching jobs for: generative AI intern jobs in India
Found 2 jobs



In [ ]:
print(response["messages"][-1].content[0]['text'])

Here's an overview of the demand for Generative AI in the industry and related intern job openings in India:

Industry Demand for Generative AI:

 The global generative AI market is experiencing rapid growth, with projections indicating an increase from USD 37.89 billion in 2025 to approximately USD 1,206.24 billion by 2035, at a compound annual growth rate (CAGR) of 36.97% from 2025 to 2034.

 Key drivers for this demand include:
  Expanding applications: Technologies like super-resolution, text-to-image conversion, and text-to-video conversion are fueling the need for generative AI across various sectors.
  Workflow modernization: Industries are increasingly adopting generative AI to automate processes, enhance efficiency, and modernize their workflows.
  Healthcare innovation: The healthcare sector is seeing significant growth in generative AI adoption, particularly with the use of 3D printing technologies for creating various products.
  Multimodal segment growth: The multimodal se

In [ ]:
user_query="tell me more about the company of the job that u have listed"

In [ ]:
response = agent.invoke({
    "messages": [{"role": "user", "content": user_query}]
},config=config)

In [ ]:
print(response["messages"][-1].content)

Here's more information about the companies offering the Generative AI intern positions:

Venura Tech

 About: Venura Tech is a next-generation tech platform that focuses on AI-guided learning, real-world projects, and structured internships. They aim to help students learn smarter, build faster, and launch their tech careers.
 Location: Bengaluru, Karnataka, India
 Offerings: They provide training in various tech domains, including Python, AI & Machine Learning, Data Science, Full Stack Web Development, Java Backend Development, and Cyber Security. Their approach emphasizes practical training, real-world projects, and expert mentorship.
 Mission: To build future-ready careers by equipping students with skills that companies actively seek.

Novartis Healthcare Private Limited

 About: Novartis in India is part of a global innovative medicines company with a legacy spanning over 250 years. They are driven by the purpose to reimagine medicine and improve and extend people's lives.
 Locat